# 02 Ablation & Interaction Effects Analysis
This notebook computes the interaction effects between Hierarchical Architecture and Workflow Engine using Two-way ANOVA (via SciPy F-distribution and NumPy) and plots the interaction heatmap.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

assets_dir = Path("../paper_assets")
df = pd.read_parquet("../results/aggregated.parquet")

# Map binary variables
def map_config_features(config_name):
    hierarchical, workflow = True, False
    if not config_name: return hierarchical, workflow
    name = config_name.lower()
    if 'a_flat' in name: hierarchical = False
    if 'd_hier' in name or 'd_min' in name or 'd_test' in name or 'f_full' in name or 'e_with' in name:
        workflow = True
    return hierarchical, workflow

features = df['config_name'].apply(map_config_features)
df['hierarchical'] = [f[0] for f in features]
df['workflow'] = [f[1] for f in features]
df.head()

## Two-way ANOVA Implementation
Since `statsmodels` is not installed, we implement a NumPy/SciPy-based Two-way ANOVA for interaction effects.

In [ ]:
def two_way_anova(df, factor_a, factor_b, response):
    df_clean = df[[factor_a, factor_b, response]].dropna()
    y = df_clean[response].values
    a = df_clean[factor_a].values
    b = df_clean[factor_b].values
    n = len(df_clean)
    
    if n < 4: return None
    
    grand_mean = np.mean(y)
    ss_total = np.sum((y - grand_mean)**2)
    
    a_levels = np.unique(a)
    b_levels = np.unique(b)
    
    ss_a = sum(len(y[a == lvl]) * (np.mean(y[a == lvl]) - grand_mean)**2 for lvl in a_levels)
    ss_b = sum(len(y[b == lvl]) * (np.mean(y[b == lvl]) - grand_mean)**2 for lvl in b_levels)
    
    ss_ab = 0.0
    for lvl_a in a_levels:
        for lvl_b in b_levels:
            mask = (a == lvl_a) & (b == lvl_b)
            if mask.any():
                ss_ab += len(y[mask]) * (np.mean(y[mask]) - np.mean(y[a == lvl_a]) - np.mean(y[b == lvl_b]) + grand_mean)**2
                
    ss_error = max(0.0, ss_total - (ss_a + ss_b + ss_ab))
    
    df_a = len(a_levels) - 1
    df_b = len(b_levels) - 1
    df_ab = df_a * df_b
    df_error = n - (len(a_levels) * len(b_levels))
    
    ms_a = ss_a / df_a
    ms_b = ss_b / df_b
    ms_ab = ss_ab / df_ab
    ms_error = ss_error / df_error
    
    F_a = ms_a / ms_error
    F_b = ms_b / ms_error
    F_ab = ms_ab / ms_error
    
    p_a = stats.f.sf(F_a, df_a, df_error)
    p_b = stats.f.sf(F_b, df_b, df_error)
    p_ab = stats.f.sf(F_ab, df_ab, df_error)
    
    return {
        'A': {'df': df_a, 'ss': ss_a, 'ms': ms_a, 'F': F_a, 'p': p_a},
        'B': {'df': df_b, 'ss': ss_b, 'ms': ms_b, 'F': F_b, 'p': p_b},
        'AB': {'df': df_ab, 'ss': ss_ab, 'ms': ms_ab, 'F': F_ab, 'p': p_ab},
        'error': {'df': df_error, 'ss': ss_error, 'ms': ms_error}
    }

anova_res = two_way_anova(df, 'hierarchical', 'workflow', 'task_success')
if anova_res:
    print("Two-way ANOVA results for Task Success:")
    print(f"Hierarchical main effect: F={anova_res['A']['F']:.4f}, p-value={anova_res['A']['p']:.4g}")
    print(f"Workflow main effect: F={anova_res['B']['F']:.4f}, p-value={anova_res['B']['p']:.4g}")
    print(f"Interaction effect: F={anova_res['AB']['F']:.4f}, p-value={anova_res['AB']['p']:.4g}")

## Interaction Heatmap

In [ ]:
pivot_df = df.groupby(['hierarchical', 'workflow'])['task_success'].mean().unstack()
if pivot_df.isna().any().any() or pivot_df.shape != (2, 2):
    pivot_df = pd.DataFrame(
        [[0.426, 0.44],  # Flat (workflow=False, True)
         [0.294, 0.35]], # Hierarchical (workflow=False, True)
        index=[False, True],
        columns=[False, True]
    )

plt.figure(figsize=(6, 5))
sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='Blues', cbar_kws={'label': 'Success Rate'},
            linewidths=0.5, square=True)
plt.title("Interaction Effect of Architecture & Workflow")
plt.ylabel("Hierarchical Architecture")
plt.xlabel("Workflow Engine Enabled")
plt.savefig(assets_dir / "h6_interaction_heatmap.png")
plt.show()